# Satellite Challenge A: Put a Price on Melissa's Farm Damage 💰

**Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and Technology, University of the West Indies.**

**The question.** Hurricane Melissa flattened vegetation around New Hope,
Westmoreland, where the eye came ashore. Relief agencies do not act on maps.
They act on money. So: **what did that vegetation damage cost, in Jamaican
dollars?**

Nobody on Earth knows this number exactly. Your team is going to build an
honest estimate, the same way a real analyst would: measure what you can
measure, research what you cannot, and be open about the range.

### How this hour works

Your team has 45 minutes to build the answer and 60 seconds to present it.

- Cells marked **🚚 JUST RUN** do the heavy coding. Run them, read what they print.
- Cells marked **✏️ YOUR CALL** hold the numbers only your team can decide.
  Every number needs a reason your team can say out loud, and the best reasons
  name a source you found: a website, a news story, a person you asked.
- The last cell prints your team's pitch. Read it to the room.

Judges reward four things: the measurement ran, the chosen numbers have named
sources, the answer is given as a range rather than fake precision, and the
pitch tells a clear story.

---

## Step 1. Measure the damaged hectares 🚚 JUST RUN

Run the two setup cells, then the measurement. It compares this storm season
against the same season one year earlier, so the change of season cancels out,
and counts the hectares that were vegetated before and lost their green. It
takes two to three minutes. Read Step 2 while it runs.

In [ ]:
# Run this once. On Google Colab it takes about a minute.
# If a package is already there, pip will say so and move on.
!pip install -q rasterio requests imageio pandas scikit-learn matplotlib pillow

print("Packages ready.")

In [ ]:
# 🚚 JUST RUN THIS CELL. Nothing to change. It is the toolbox for the whole course.
# ============================================================================
#  JAMAICA EARTH OBSERVATION TOOLKIT
#  Run this cell in every session. It sets up the connection to the satellite
#  archive and defines the handful of functions the whole course uses.
# ============================================================================
import os, math, json, time, warnings
warnings.filterwarnings("ignore")

# GDAL reads the satellite files straight off Amazon's servers over the
# internet. These settings tell it how to behave: no login needed, do not list
# whole directories, retry if the network hiccups.
os.environ.update({
    "AWS_NO_SIGN_REQUEST": "YES",
    "GDAL_HTTP_UNSAFESSL": "YES",
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MAX_RETRY": "5",
    "GDAL_HTTP_RETRY_DELAY": "2",
})

import requests, numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from rasterio.warp import Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.vrt import WarpedVRT
from PIL import Image, ImageDraw

STAC_URL = "https://earth-search.aws.element84.com/v1/search"

def _stac_post(url, body, timeout=60, tries=4):
    """POST to the archive, retrying politely if the server is having a moment."""
    for attempt in range(tries):
        try:
            r = requests.post(url, json=body, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))    # 2 s, 4 s, 6 s between tries


# House style for every chart in this course.
CYAN, INK, SAND = "#00b8d4", "#12232e", "#e0a458"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 13,
    "axes.titleweight": "bold", "figure.facecolor": "white",
})

# Places in Jamaica used through the course, as [west, south, east, north].
PLACES = {
    "kingston":     [-76.86, 17.93, -76.80, 18.02],
    "black_river":  [-77.90, 17.96, -77.78, 18.08],
    "negril":       [-78.375, 18.25, -78.320, 18.36],
    "montego_bay":  [-77.97, 18.44, -77.88, 18.51],
    "new_hope":     [-78.20, 18.13, -78.08, 18.24],
    "st_elizabeth": [-77.75, 18.00, -77.65, 18.10],
    "portland":     [-76.45, 18.10, -76.32, 18.20],
    "jamaica":      [-78.45, 17.66, -76.15, 18.55],
}

def search_scenes(bbox, start, end, max_cloud=30, limit=50, sort_by="eo:cloud_cover",
                  min_cloud=None, descending=False):
    """Ask the archive which Sentinel-2 pictures exist over a box and a date range.

    Returns a list of STAC 'items'. Each item is a dictionary of metadata plus
    links to the actual image files. Nothing is downloaded yet.

    Set `min_cloud` when you deliberately want a cloudy scene, which is useful
    for testing that your cloud masking actually works.
    """
    cloud_filter = {"lt": max_cloud}
    if min_cloud is not None:
        cloud_filter["gt"] = min_cloud
    query = {
        "collections": ["sentinel-2-l2a"],
        "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": cloud_filter},
        "limit": limit,
        "sortby": [{"field": f"properties.{sort_by}",
                    "direction": "desc" if descending else "asc"}],
    }
    r = _stac_post(STAC_URL, query, timeout=60)
    return r.json()["features"]

def search_all(bbox, start, end, max_cloud=100, page_size=100, max_pages=20):
    """Every matching scene, not just the first page.

    The archive hands back at most 200 results per request and does not warn you
    that it stopped. This follows the 'next' link until the results run out.
    """
    body = {
        "collections": ["sentinel-2-l2a"], "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": {"lt": max_cloud}},
        "limit": page_size,
        "sortby": [{"field": "properties.datetime", "direction": "asc"}],
    }
    items, url, pages, matched = [], STAC_URL, 0, None
    while url and pages < max_pages:
        r = _stac_post(url, body, timeout=90)
        j = r.json()
        items += j.get("features", [])
        matched = j.get("context", {}).get("matched", matched)
        nxt = [l for l in j.get("links", []) if l.get("rel") == "next"]
        pages += 1
        if not nxt:
            break
        url = nxt[0]["href"]
        body = nxt[0].get("body", body)
    if matched and len(items) < matched:
        print(f"Warning: got {len(items)} of {matched}. Raise max_pages.")
    return items

def make_grid(bbox, metres=20):
    """Define a fixed grid of pixels over a box, in plain latitude and longitude.

    Every image we read gets warped onto this same grid. That is what lets us
    subtract a November picture from an October one pixel by pixel, even when
    the two came from different satellite tiles in different map projections.
    """
    lon0, lat0, lon1, lat1 = bbox
    shrink = math.cos(math.radians((lat0 + lat1) / 2))
    width  = int(round((lon1 - lon0) * 111320 * shrink / metres))
    height = int(round((lat1 - lat0) * 110540 / metres))
    transform = transform_from_bounds(lon0, lat0, lon1, lat1, width, height)
    return {"width": width, "height": height, "transform": transform,
            "metres": metres, "bbox": bbox,
            "pixel_hectares": (metres * metres) / 10000.0}

def read_band(item, band, grid, resampling=Resampling.bilinear):
    """Read one colour band of one scene onto our grid. Returns raw integers."""
    with rasterio.open(item["assets"][band]["href"]) as src:
        with WarpedVRT(src, crs="EPSG:4326", transform=grid["transform"],
                       width=grid["width"], height=grid["height"],
                       resampling=resampling) as vrt:
            return vrt.read(1)

def read_reflectance(item, band, grid):
    """Read a band and convert to reflectance (0 to 1). Divide by 10000."""
    return read_band(item, band, grid).astype("float32") / 10000.0

# Scene Classification Layer codes that mean 'this pixel is usable'.
# 4 vegetation, 5 bare soil, 6 water, 7 low-probability cloud, 11 snow/ice.
CLEAR_CODES = [4, 5, 6, 7, 11]

def clear_mask(item, grid):
    """True where the pixel is usable, False where it is cloud, shadow or edge."""
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return np.isin(scl, CLEAR_CODES)

def check_coverage(item, grid):
    """How much of OUR area this scene actually covers, and how much is clear.

    The cloud percentage in the metadata describes the whole 110 km tile. It
    says nothing about your study area. Always check your own box.
    """
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return {"covered": float((scl > 0).mean()),
            "clear": float(np.isin(scl, CLEAR_CODES).mean())}

def best_scene(items, grid, min_covered=0.95, min_clear=0.60, check_n=8):
    """Walk down the candidate list and return the first scene that is genuinely
    good over our box, not just good on paper."""
    for item in items[:check_n]:
        try:
            c = check_coverage(item, grid)
        except Exception:
            continue
        if c["covered"] >= min_covered and c["clear"] >= min_clear:
            item["_coverage"] = c
            return item
    return None

def composite(items, grid, bands, max_scenes=12, min_clear=0.10, verbose=True):
    """Stack several cloud-masked scenes and take the middle value per pixel.

    One picture of Jamaica almost always has cloud somewhere. Stack ten and take
    the median and the clouds disappear, because cloud is bright and rare while
    the ground underneath is consistent.
    """
    stacks = {b: [] for b in bands}
    used = []
    for item in items:
        if len(used) >= max_scenes:
            break
        try:
            clear = clear_mask(item, grid)
            if clear.mean() < min_clear:
                continue
            for b in bands:
                a = read_reflectance(item, b, grid)
                a[~clear] = np.nan
                a[a <= 0] = np.nan
                stacks[b].append(a)
            used.append(item["properties"]["datetime"][:10])
        except Exception:
            continue
    if not used:
        raise RuntimeError("No usable scenes found. Widen the dates or raise max_cloud.")
    if verbose:
        print(f"Composite built from {len(used)} scenes: {', '.join(sorted(used))}")
    out = {b: np.nanmedian(np.stack(v), axis=0) for b, v in stacks.items()}
    out["_dates"] = sorted(used)
    return out

def normalized_difference(a, b):
    """(a - b) / (a + b). The workhorse formula behind every index in this course."""
    return (a - b) / (a + b + 1e-10)

def stretch(rgb, low=2, high=98):
    """Rescale each colour channel so the picture is bright enough to look at."""
    out = np.zeros_like(rgb, dtype="float32")
    for i in range(rgb.shape[2]):
        band = rgb[:, :, i]
        p1, p2 = np.nanpercentile(band, [low, high])
        out[:, :, i] = np.clip((band - p1) / (p2 - p1 + 1e-9), 0, 1)
    return np.nan_to_num(out)

def show(image, title="", cmap=None, vmin=None, vmax=None, bar=False, size=(9, 8)):
    """Draw an array on screen with sensible defaults."""
    fig, ax = plt.subplots(figsize=size)
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if bar:
        fig.colorbar(im, ax=ax, shrink=0.75)
    plt.tight_layout(); plt.show()

def area_hectares(mask, grid):
    """Convert a True/False mask into hectares on the ground."""
    return float(np.nansum(mask)) * grid["pixel_hectares"]

def label_frame(image_uint8, text):
    """Stamp a label bar onto one animation frame, so every frame says what it is."""
    img = Image.fromarray(image_uint8)
    draw = ImageDraw.Draw(img)
    bar = min(14 + 8 * len(text), img.width)
    draw.rectangle([0, 0, bar, 24], fill=(0, 0, 0))
    draw.text((7, 6), text, fill=(255, 255, 255))
    return np.array(img)

def save_gif(frames, path, ms=900):
    """Write labelled frames out as an animated GIF that loops forever."""
    import imageio.v2 as imageio
    imageio.mimsave(path, frames, duration=ms, loop=0)
    print(f"Saved {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {len(frames)} frames)")

def show_gif(path):
    """Play a GIF inside the notebook."""
    try:
        from IPython.display import Image as _Gif, display
        display(_Gif(filename=path))
    except Exception:
        print("Open the file from the folder panel on the left to watch it.")

print("Toolkit loaded. Study areas available:", ", ".join(PLACES))

In [ ]:
# 🚚 JUST RUN. Measures the damage around New Hope, where Melissa's eye landed.
box  = PLACES["new_hope"]
grid = make_grid(box, metres=30)

# same weeks of the year, one year apart, so the season cancels out
before_scenes = search_scenes(box, "2024-11-01", "2024-12-20", max_cloud=60, limit=30)
after_scenes  = search_scenes(box, "2025-11-01", "2025-12-20", max_cloud=60, limit=30)

bands = ["red", "nir"]
before = composite(before_scenes, grid, bands, max_scenes=5)
after  = composite(after_scenes,  grid, bands, max_scenes=5)

ndvi_before = normalized_difference(before["nir"], before["red"])
ndvi_after  = normalized_difference(after["nir"],  after["red"])
change = ndvi_after - ndvi_before

# damaged = was vegetated before, and lost a lot of green
damaged = (ndvi_before > 0.25) & (change < -0.20)
damaged_ha = area_hectares(damaged, grid)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(change, cmap="RdYlGn", vmin=-0.5, vmax=0.5)
ax[0].set_title("NDVI change across Melissa (red = lost green)"); ax[0].axis("off")
ax[1].imshow(damaged, cmap="Reds")
ax[1].set_title(f"Damaged: {damaged_ha:,.0f} hectares"); ax[1].axis("off")
plt.show()

print(f"MEASURED: {damaged_ha:,.0f} hectares of vegetation damage around New Hope")

---

## Step 2. Your call: the numbers your team must find and defend ✏️

The satellite measured hectares. Money needs three more numbers, and no
satellite can supply them. Your phones can. Split the research between
teammates.

| Number | Where to look |
|---|---|
| What share of the damaged land is farmland? | Look at the change map above, and at the area on Google Maps satellite view |
| What is a hectare of Jamaican farmland worth per year, in Jamaican dollars? | Search for crop earnings per hectare in Jamaica: RADA (the Rural Agricultural Development Authority), news reports, Ministry of Agriculture figures |
| How many years before the land produces again? | Search for crop recovery time after a hurricane: bananas take about a year, tree crops far longer |

You will not find one true value. That is the point. Pick a low, a best guess,
and a high, and write down where each came from.

In [ ]:
# ✏️ YOUR CALL. Change every number below, and give every number its reason.

team_name = "____"                      # your team name

farm_share = 0.40                       # share of damaged land that is farmland (0 to 1)
value_low_jmd  =  150_000               # J$ one hectare earns per year, low end
value_best_jmd =  300_000               # J$ per hectare per year, your best guess
value_high_jmd =  600_000               # J$ per hectare per year, high end
recovery_years = 2                      # years before the land earns again

reasons = {
    "farm_share":  "____",              # e.g. "Google Maps shows the area is mostly fields"
    "value_per_hectare": "____",        # name the site, report or person
    "recovery_years": "____",           # what crop did you assume, and why
}

for k, v in reasons.items():
    if "____" in v:
        print(f"⚠️  Fill in your reason for {k}. A number without a reason scores nothing.")

---

## Step 3. Your formula and your answer 🚚 JUST RUN

The formula multiplies what you measured by what you researched:

**cost = damaged hectares × farm share × value per hectare per year × years of recovery**

If your team's logic differs, say so in the pitch. That is allowed. Hiding it
is not.

In [ ]:
# 🚚 JUST RUN. Your estimate, as a range.
def cost_jmd(value_per_ha):
    return damaged_ha * farm_share * value_per_ha * recovery_years

low, best, high = cost_jmd(value_low_jmd), cost_jmd(value_best_jmd), cost_jmd(value_high_jmd)

print(f"Team {team_name} estimates the cost of Melissa's vegetation damage around New Hope:")
print(f"  Low:   J$ {low/1e6:,.0f} million")
print(f"  Best:  J$ {best/1e6:,.0f} million")
print(f"  High:  J$ {high/1e6:,.0f} million")

---

## Step 4. Your 60-second pitch 🚚 JUST RUN

Run the cell. It writes your script. One teammate reads it, another holds up
the change map, a third fields the one question judges always ask: *which of
your numbers are you least sure about?* Decide your answer to that before you
stand up.

In [ ]:
# 🚚 JUST RUN. Your pitch, written from your own numbers.
print(f"""
We are team {team_name}.

The satellite measured {damaged_ha:,.0f} hectares of vegetation damage around
New Hope, comparing this storm season against the same weeks last year, so the
season could not fool us.

We estimate {farm_share:.0%} of that land is farmland, because {reasons['farm_share']}.
We value a farmed hectare at J$ {value_best_jmd:,} per year, based on {reasons['value_per_hectare']}.
We assume {recovery_years} year(s) of lost production, because {reasons['recovery_years']}.

Our estimate: J$ {best/1e6:,.0f} million, and honestly somewhere between
J$ {low/1e6:,.0f} million and J$ {high/1e6:,.0f} million.
""")

---

*Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and
Technology, University of the West Indies.*

*Satellite Data Analysis for Jamaica. Built with free, open data: Sentinel-2
from the European Space Agency (ESA), hosted by Amazon; NASA POWER climate
records. No accounts, no fees, no permission needed.*